In [1]:
import os
import pwd
import sys
import numpy as np
import pandas as pd
from random import randrange

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

username = pwd.getpwuid(os.getuid()).pw_name
hadoopFS = os.getenv('HADOOP_FS', None)
groupName = "H1"

print(f"username={username}")
print(f"hadoopFS={hadoopFS}")
print(f"group={groupName}")

username=fjelstad
hadoopFS=hdfs://iccluster061.iccluster.epfl.ch:9000
group=H1


In [2]:
spark = (SparkSession\
            .builder
            .appName(username + '-assignment-2')
            .config('spark.ui.port', randrange(4050, 4450, 5))
            .config("spark.executorEnv.PYTHONPATH", ":".join(sys.path))
            .config('spark.jars',
                    f'{hadoopFS}/data/com-490/jars/iceberg-spark-runtime-3.5_2.13-1.6.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/sedona-spark-shaded-3.5_2.13-1.7.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/geotools-wrapper-1.7.1-28.5.jar'
            )
            .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
            .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
            .config('spark.sql.catalog.iceberg.type', 'hadoop')
            .config('spark.sql.catalog.iceberg.warehouse', f'{hadoopFS}/data/com-490/silver/')
            .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
            .config('spark.sql.catalog.spark_catalog.type', 'hadoop')
            .config('spark.sql.catalog.spark_catalog.warehouse', f'{hadoopFS}/user/{username}/assignment-3/warehouse')
            .config("spark.sql.warehouse.dir", f'{hadoopFS}/user/{username}/assignment-3/spark/warehouse')
            .config("spark.executor.memory", "6g")
            .config("spark.executor.cores", "4")
            .config("spark.executor.instances", "4")
        ).master('yarn').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 20:32:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
spark.sparkContext

<SparkContext master=yarn appName=fjelstad-assignment-2>

In [4]:
from sedona.spark import SedonaContext
spark = SedonaContext.create(spark)

## Config

In [5]:
# Paths
BASE = f"{hadoopFS}/user/groups/com-490/H1/final/v1"
TIMETABLE_PATH = f"{BASE}/timetable_february.parquet"
TRANSFERS_PATH = f"{BASE}/transfers.parquet"
LOOKUP_PATH = f"{BASE}/route_demo_outputs/lookup/route_demo_lookup_february.parquet"

# Routing week
WEEK_START = "2026-02-02"
WEEK_END = "2026-02-08"

# Walking rules (from spec FAQ)
WALKING_SPEED_M_PER_MIN = 50
MIN_TRANSFER_SECONDS = 120
MAX_WALK_DISTANCE_M = 500 

## Build walking graph (stop_to_stop)

In [6]:
# Read trip_stop_events
spark.read.parquet(f"{BASE}/trip_stop_events.parquet") \
     .createOrReplaceTempView("base_trips")

# Merge platforms, one (lat, lon) per station
spark.sql("""
    SELECT 
        cleaned_stop_id AS stop_id,
        FIRST(stop_name) AS stop_name,
        AVG(stop_lat) AS stop_lat,
        AVG(stop_lon) AS stop_lon
    FROM base_trips
    GROUP BY cleaned_stop_id
""").createOrReplaceTempView("region_stops")

# All stop pairs within 500m
stop_to_stop_df = spark.sql("""
    SELECT
        a.stop_id AS a_stop_id,
        b.stop_id AS b_stop_id,
        ST_DistanceSphere(
            ST_Point(a.stop_lon, a.stop_lat),
            ST_Point(b.stop_lon, b.stop_lat)
        ) AS distance_m
    FROM region_stops a
    JOIN region_stops b ON a.stop_id < b.stop_id
    WHERE ST_DistanceSphere(
              ST_Point(a.stop_lon, a.stop_lat),
              ST_Point(b.stop_lon, b.stop_lat)
          ) <= 500
""")

stop_to_stop_df.write.mode("overwrite").parquet(f"{BASE}/stop_to_stop.parquet")
print(f"Stop pairs: {stop_to_stop_df.count()}")
stop_to_stop_df.show(5)

Stop pairs: 1249


+---------+---------+------------------+
|a_stop_id|b_stop_id|        distance_m|
+---------+---------+------------------+
|  8501210|  8595521|196.03862402782403|
|  8592210|  8595521|362.56133761674613|
|  8592219|  8595521| 464.1552787837826|
|  8501211|  8595521|348.96178304282336|
|  8591997|  8592048|360.14633239903293|
+---------+---------+------------------+
only showing top 5 rows



In [7]:
# Check for duplicates
stop_to_stop_df.groupBy("a_stop_id", "b_stop_id") \
    .count() \
    .filter("count > 1") \
    .show()

+---------+---------+-----+
|a_stop_id|b_stop_id|count|
+---------+---------+-----+
+---------+---------+-----+



## Load and prepare routing data

In [8]:
import pandas as pd
import math
from collections import defaultdict

### Load timetable

In [9]:
# Load February timetable
timetable_pdf = (
    spark.read.parquet(TIMETABLE_PATH)
    .filter(F.col("arrival_timestamp") >= F.lit(WEEK_START))
    .filter(F.col("arrival_timestamp") < F.lit(f"{WEEK_END} 23:59:59"))
    .toPandas()
)

# Ensure timestamps are pandas datetimes
timetable_pdf["arrival_timestamp"]   = pd.to_datetime(timetable_pdf["arrival_timestamp"])
timetable_pdf["departure_timestamp"] = pd.to_datetime(timetable_pdf["departure_timestamp"])

# Extract BPUIC
timetable_pdf["cleaned_stop_id"] = timetable_pdf["stop_id"].str.split(":").str[0]

print(f"Timetable rows: {len(timetable_pdf):,}")
print(f"Unique trips: {timetable_pdf['trip_id'].nunique():,}")
print(f"Unique stops: {timetable_pdf['cleaned_stop_id'].nunique():,}")

Timetable rows: 784,078
Unique trips: 23,709
Unique stops: 381


### Load walking graph

In [10]:
# The stop_to_stop parquet
stop_to_stop_pdf = spark.read.parquet(f"{BASE}/stop_to_stop.parquet").toPandas()

print(f"Walking edges: {len(stop_to_stop_pdf):,}")
print(f"Distance range: {stop_to_stop_pdf['distance_m'].min():.0f}m - {stop_to_stop_pdf['distance_m'].max():.0f}m")
print(f"Median distance: {stop_to_stop_pdf['distance_m'].median():.0f}m")

Walking edges: 1,249
Distance range: 27m - 500m
Median distance: 352m


### Build trips dict

In [11]:
# Service day: shift by 4h so midnight-crossing trips stay grouped together
# TODO: IMPROVE THIS WITH TARGET DATE IN routing_data.ipynb
timetable_pdf["service_day"] = (
    (timetable_pdf["departure_timestamp"] - pd.Timedelta(hours=4))
    .dt.date.astype(str)
)
# Unique key per trip-instance
timetable_pdf["trip_key"] = (
    timetable_pdf["trip_id"] + "#" + timetable_pdf["service_day"]
)

# Build trips dict
sorted_pdf = timetable_pdf.sort_values(["trip_key", "stop_sequence"])
trips = {}
for trip_key, group in sorted_pdf.groupby("trip_key", sort=False):
    trips[trip_key] = [
        (str(row.cleaned_stop_id),
         row.arrival_timestamp,
         row.departure_timestamp,
         int(row.stop_sequence))
        for row in group.itertuples(index=False)
    ]

print(f"Trips: {len(trips):,}")

# Sanity check
sample = next(iter(trips))
arrivals = [s[1] for s in trips[sample]]
is_monotonic = all(arrivals[i] <= arrivals[i+1] for i in range(len(arrivals)-1))
print(f"Sample trip monotonic: {is_monotonic}")

Trips: 56,098
Sample trip monotonic: True


### Build trip_stop_index dict for fast lookup

In [12]:
# Index: (trip_key, stop_id) -> position in trip's stop list
trip_stop_index = {}
for trip_key, stops in trips.items():
    for idx, (stop_id, _, _, _) in enumerate(stops):
        trip_stop_index[(trip_key, stop_id)] = idx

print(f"Trip-stop index entries: {len(trip_stop_index):,}")

Trip-stop index entries: 778,866


### Build routes_at_stop dict

In [13]:
# Index: stop_id -> list of trip_keys passing through
routes_at_stop = defaultdict(set)
for trip_key, stops in trips.items():
    for stop_id, _, _, _ in stops:
        routes_at_stop[stop_id].add(trip_key)
routes_at_stop = {s: list(ts) for s, ts in routes_at_stop.items()}

print(f"Stops with trips: {len(routes_at_stop):,}")

sample_stop = next(iter(routes_at_stop))
print(f"Stop {sample_stop} has {len(routes_at_stop[sample_stop])} trips passing through")

Stops with trips: 381
Stop 8530749 has 8987 trips passing through


### Build footpaths dict

In [14]:
# Given stop get (neighbor_stop, walk_seconds) tuples
footpaths = defaultdict(list)
for row in stop_to_stop_pdf.itertuples(index=False):
    if row.distance_m > MAX_WALK_DISTANCE_M:
        continue
    walk_sec = max(
        MIN_TRANSFER_SECONDS,
        math.ceil(row.distance_m / WALKING_SPEED_M_PER_MIN * 60)
    )
    a, b = str(row.a_stop_id), str(row.b_stop_id)
    footpaths[a].append((b, walk_sec))
    footpaths[b].append((a, walk_sec))   # symmetri

footpaths = dict(footpaths)
print(f"Stops with footpaths: {len(footpaths):,}")

sample = next(iter(footpaths))
print(f"Stop {sample} neighbors: {footpaths[sample][:3]}")

Stops with footpaths: 382
Stop 8501210 neighbors: [('8595521', 236), ('8595937', 478), ('8501211', 505)]


### Load delay prediction lookup

In [15]:
# Calibrated delay quantiles per (trip, stop, day) from the predictive model
lookup_pdf = (
    spark.read.parquet(LOOKUP_PATH)
    .filter(F.col("operating_day") >= F.lit(WEEK_START))
    .filter(F.col("operating_day") <= F.lit(WEEK_END))
    .toPandas()
)

print(f"Lookup rows: {len(lookup_pdf):,}")
print(f"Columns: {list(lookup_pdf.columns)}")

Lookup rows: 784,078
Columns: ['operating_day', 'trip_id', 'bpuic', 'stop_name', 'scheduled_arrival_ts', 'line_text', 'transport_clean', 'pred_delay', 'chosen_calibration_level_q90', 'q01', 'q05', 'q10', 'q20', 'q35', 'q50', 'q65', 'q80', 'q90', 'q95', 'q99', 'line_seen_in_training', 'line_id_source']


### Build lookup dict

In [16]:
# Quantile columns from the calibrated delay model
QUANTILE_COLS = ["q01", "q05", "q10", "q20", "q35", "q50",
                 "q65", "q80", "q90", "q95", "q99"]

# Used for computing the delay probalility
lookup = {}
for row in lookup_pdf.itertuples(index=False):
    key = (str(row.trip_id), int(row.bpuic), row.operating_day)
    lookup[key] = {q: getattr(row, q) for q in QUANTILE_COLS}
    lookup[key]["pred_delay"] = row.pred_delay

print(f"Lookup entries: {len(lookup):,}")

Lookup entries: 778,868


## Routing algorithm

In [17]:
from compare.raptor import Raptor, build_routes_from_trips

routes, routes_at_stop = build_routes_from_trips(trips)
print(f"Routes: {len(routes):,}  (vs {len(trips):,} trips)")

planner = Raptor(routes, routes_at_stop, footpaths)

Routes: 340  (vs 56,098 trips)


In [18]:
%%time
T = pd.Timestamp("2026-02-04 09:00:00")

journeys = planner.query_range(
    source="8501214", target="8592050",
    arrive_before=T, window_hours=2.0, step_minutes=10,
)

journeys = planner.query_range(
    source="8501214", target="8592050",
    arrive_before=T, window_hours=2.0, step_minutes=10,
    lookup=lookup, min_confidence=0.8,
)

print(f"Found {len(journeys)} robust journeys:\n")
for j in journeys[:5]:
    conf = planner.journey_confidence(j, lookup, T)
    print(f"=== Confidence {conf:.1%} ===")
    print(j)
    print()

Found 10 robust journeys:

=== Confidence 98.4% ===
Journey: dep 2026-02-04 08:32:00 -> arr 2026-02-04 08:49:54, 0 transfers
  08:32 8501214 -> 08:44 8591818 [1630.TA.91-m1-j26-1.3.H#2026-02-04]
  08:44 8591818 -> 08:49 8592050 [WALK]

=== Confidence 100.0% ===
Journey: dep 2026-02-04 08:24:00 -> arr 2026-02-04 08:42:54, 0 transfers
  08:24 8501214 -> 08:37 8591818 [1260.TA.91-m1-j26-1.3.H#2026-02-04]
  08:37 8591818 -> 08:42 8592050 [WALK]

=== Confidence 95.6% ===
Journey: dep 2026-02-04 08:10:00 -> arr 2026-02-04 08:31:25, 1 transfers
  08:10 8501214 -> 08:15 8530749 [2431.TA.91-m1-j26-1.7.R#2026-02-04]
  08:15 8530749 -> 08:17 8501118 [WALK]
  08:24 8501118 -> 08:29 8501120 [1687.TA.91-33-C-j26-1.94.R#2026-02-04]
  08:29 8501120 -> 08:31 8592050 [WALK]

=== Confidence 100.0% ===
Journey: dep 2026-02-04 08:02:00 -> arr 2026-02-04 08:19:54, 0 transfers
  08:02 8501214 -> 08:14 8591818 [1437.TA.91-m1-j26-1.3.H#2026-02-04]
  08:14 8591818 -> 08:19 8592050 [WALK]

=== Confidence 100.0% 

In [19]:
%%time
import pandas as pd

test_cases = [
    ("Morning", "8501214", "8592050", pd.Timestamp("2026-02-04 09:00:00")),
    ("Afternoon", "8501214", "8592050", pd.Timestamp("2026-02-04 17:00:00")),
    ("Midnight crossing", "8501214", "8592050", pd.Timestamp("2026-02-05 00:30:00")),
    ("Saturday", "8501214", "8592050", pd.Timestamp("2026-02-07 14:00:00")),
]

for name, src, tgt, before in test_cases:
    print(f"\n=== {name} ===")
    journeys = planner.query_range(
        source=src, target=tgt,
        arrive_before=before,
        window_hours=1.5, step_minutes=10,
    )
    print(f"Found {len(journeys)} journeys")
    for j in journeys[:3]:
        conf = planner.journey_confidence(j, lookup, before)
        print(f"  dep={j.departure.strftime('%H:%M')} "
              f"arr={j.arrival.strftime('%H:%M')} "
              f"transfers={j.num_transfers} "
              f"conf={conf:.1%}")


=== Morning ===
Found 8 journeys
  dep=08:40 arr=08:58 transfers=1 conf=49.6%
  dep=08:32 arr=08:49 transfers=0 conf=98.4%
  dep=08:24 arr=08:42 transfers=0 conf=100.0%

=== Afternoon ===
Found 8 journeys
  dep=16:40 arr=16:58 transfers=1 conf=70.9%
  dep=16:32 arr=16:49 transfers=0 conf=94.7%
  dep=16:24 arr=16:42 transfers=0 conf=96.9%

=== Midnight crossing ===
Found 7 journeys
  dep=00:03 arr=00:20 transfers=0 conf=100.0%
  dep=23:53 arr=00:10 transfers=1 conf=87.9%
  dep=23:43 arr=00:00 transfers=0 conf=100.0%

=== Saturday ===
Found 8 journeys
  dep=13:40 arr=13:58 transfers=1 conf=87.0%
  dep=13:32 arr=13:49 transfers=0 conf=100.0%
  dep=13:24 arr=13:42 transfers=0 conf=100.0%
CPU times: user 911 ms, sys: 12.4 ms, total: 923 ms
Wall time: 919 ms
